In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q transformers accelerate sentencepiece bitsandbytes wandb huggingface_hub

In [3]:
import os
import json
import time
import torch
import pandas as pd

from pathlib import Path
from huggingface_hub import login

import wandb

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
from huggingface_hub import login
login("<HF_TOKEN>")

wandb.login(key="<WANDB_API_KEY>")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nishkala-aistuff (nishkala-aistuff-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

BENCHMARK_PATH = (
    f"{PROJECT_ROOT}/eval/llm_only/benchmarks/llm_only_v1.json"
)

PROMPTS_DIR = (
    f"{PROJECT_ROOT}/eval/llm_only/prompts"
)

RUNS_DIR = (
    f"{PROJECT_ROOT}/eval/llm_only/runs/prompt_comparison"
)

OUTPUT_CSV = (
    f"{PROJECT_ROOT}/eval/llm_only/scores/prompt_augmentation_stats.csv"
)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

TEMPERATURE = 0.0
MAX_NEW_TOKENS = 300

WANDB_PROJECT = "ARIA-Lite-Eval"
WANDB_GROUP = "prompt_augmentation_v1"

In [6]:
os.makedirs(RUNS_DIR, exist_ok=True)

In [7]:
with open(BENCHMARK_PATH, "r") as f:
    benchmark = json.load(f)

print(f"Loaded {len(benchmark)} benchmark questions")

Loaded 4 benchmark questions


In [8]:
prompt_files = sorted(
    Path(PROMPTS_DIR).glob("*.txt")
)

print("Prompts Found:")

for p in prompt_files:
    print("-", p.name)

Prompts Found:
- prompt_v1.txt
- prompt_v2_concise.txt
- prompt_v3_grounded.txt
- prompt_v4_rubric.txt


In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model Loaded


In [10]:
def generate_answer(prompt_text):

    messages = [
        {
            "role": "user",
            "content": prompt_text
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=TEMPERATURE
    )

    latency = time.time() - start

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response, latency

In [11]:
all_stats = []

wandb.init(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    name="qwen3b_prompt_comparison"
)

for prompt_file in prompt_files:

    prompt_name = prompt_file.stem

    print("\n")
    print("=" * 80)
    print(prompt_name)
    print("=" * 80)

    prompt_output_dir = (
        f"{RUNS_DIR}/{prompt_name}"
    )

    os.makedirs(
        prompt_output_dir,
        exist_ok=True
    )

    with open(prompt_file, "r") as f:
        prompt_template = f.read()

    for item in benchmark:

        query_id = item["query_id"]

        context = "\n\n".join([
            s["text"]
            for s in item["sections"]
        ])

        prompt = prompt_template.format(
            query=item["query"],
            sections=context
        )

        response, latency = generate_answer(prompt)

        output_json = {
            "query_id": query_id,
            "prompt_name": prompt_name,
            "model": MODEL_ID,
            "response": response
        }

        save_path = (
            f"{prompt_output_dir}/{query_id}.json"
        )

        with open(save_path, "w") as f:
            json.dump(
                output_json,
                f,
                indent=2
            )

        token_count = len(
            tokenizer.encode(response)
        )

        row = {
            "query_id": query_id,
            "prompt_name": prompt_name,
            "latency_sec": latency,
            "response_tokens": token_count
        }

        all_stats.append(row)

        wandb.log(row)

        print(
            f"{query_id} | "
            f"{token_count} tokens | "
            f"{latency:.2f}s"
        )

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.




prompt_v1


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Q1 | 254 tokens | 51.89s
Q2 | 300 tokens | 37.38s
Q3 | 149 tokens | 17.41s
Q4 | 210 tokens | 24.90s


prompt_v2_concise
Q1 | 160 tokens | 18.55s
Q2 | 105 tokens | 20.12s
Q3 | 153 tokens | 18.56s
Q4 | 184 tokens | 21.67s


prompt_v3_grounded
Q1 | 300 tokens | 34.60s
Q2 | 171 tokens | 20.40s
Q3 | 173 tokens | 20.76s
Q4 | 199 tokens | 24.23s


prompt_v4_rubric
Q1 | 300 tokens | 34.31s
Q2 | 300 tokens | 35.77s
Q3 | 300 tokens | 34.10s
Q4 | 300 tokens | 45.25s


In [12]:
stats_df = pd.DataFrame(all_stats)

stats_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Saved:", OUTPUT_CSV)

stats_df.head()

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/llm_only/scores/prompt_augmentation_stats.csv


,query_id,prompt_name,latency_sec,response_tokens
0,Q1,prompt_v1,51.893861,254
1,Q2,prompt_v1,37.378967,300
2,Q3,prompt_v1,17.406169,149
3,Q4,prompt_v1,24.901402,210
4,Q1,prompt_v2_concise,18.553724,160


In [13]:
summary = (
    stats_df
    .groupby("prompt_name")
    .agg({
        "latency_sec":"mean",
        "response_tokens":"mean"
    })
)

summary

,latency_sec,response_tokens
prompt_name,,
prompt_v1,32.895100,228.25
prompt_v2_concise,19.727664,150.50
prompt_v3_grounded,24.997912,210.75
prompt_v4_rubric,37.356836,300.00


In [14]:
wandb.log(
    {
        "prompt_summary":
        wandb.Table(
            dataframe=summary.reset_index()
        )
    }
)

wandb.finish()

latency_sec,█▅▁▃▁▂▁▂▄▂▂▂▄▅▄▇
response_tokens,▆█▃▅▃▁▃▄█▃▃▄████
latency_sec,45.25206
prompt_name,prompt_v4_rubric
query_id,Q4
response_tokens,300
